# Noised Data Feature Extraction Pipeline

In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import cv2
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from skimage.feature import graycomatrix, graycoprops

SEED = 67
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
MODEL_PATH = "../models"
FEAT_PATH = "../features/ensemble_noise"

VARIANTS = ['salt_pepper_0.03' , 'salt_pepper_0.05' , 'salt_pepper_0.07' , 'gaussian' , 'masking',]

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 38

for v in VARIANTS:
    os.makedirs(os.path.join(FEAT_PATH, v), exist_ok=True)
    print(f"Ready : {FEAT_PATH}/{v}")

Ready : ../features/ensemble_noise/salt_pepper_0.03
Ready : ../features/ensemble_noise/salt_pepper_0.05
Ready : ../features/ensemble_noise/salt_pepper_0.07
Ready : ../features/ensemble_noise/gaussian
Ready : ../features/ensemble_noise/masking


In [3]:
transform = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)),transforms.ToTensor(),])

# Loading Models

In [4]:
mobilenet = models.mobilenet_v3_small()
in_features = mobilenet.classifier[3].in_features
mobilenet.classifier[3] = nn.Linear(in_features, NUM_CLASSES)

mobilenet.load_state_dict(torch.load(os.path.join(MODEL_PATH, "mobilenetv3.pth"), map_location=device))
mobilenet = mobilenet.to(device).eval()
print("MobileNetV3 loaded")

MobileNetV3 loaded


In [5]:
googlenet = models.googlenet(aux_logits=False)
in_features = googlenet.fc.in_features
googlenet.fc = nn.Linear(in_features, NUM_CLASSES)

googlenet.load_state_dict(torch.load(os.path.join(MODEL_PATH, "googlenet.pth"), map_location=device))
googlenet = googlenet.to(device).eval()
print("GoogLeNet loaded")

GoogLeNet loaded


/home/rishi_677/Plant_Disease_DAC_Project/venv/lib/python3.12/site-packages/torchvision/models/googlenet.py:47: FutureWarning: The default weight initialization of GoogleNet will be changed in future releases of torchvision. If you wish to keep the old behavior (which leads to long initialization times due to scipy/scipy#11299), please set init_weights=True.
  warnings.warn(


In [6]:
convnext = models.convnext_small()
in_features = convnext.classifier[2].in_features
convnext.classifier[2] = nn.Linear(in_features, NUM_CLASSES)

convnext.load_state_dict(torch.load(os.path.join(MODEL_PATH, "convnext_small.pth"), map_location=device))
convnext = convnext.to(device).eval()
print("ConvNeXt loaded")

ConvNeXt loaded


In [7]:
def get_glcm_features(img):
  img = img.permute(1, 2, 0).cpu().numpy()
  img = (img * 255).astype(np.uint8)
  gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

  glcm = graycomatrix(gray, distances =[1, 2], angles = [0, np.pi/4, np.pi/2, 3*np.pi/4], levels = 256, symmetric = True, normed = True)

  contrast = graycoprops(glcm, "contrast").flatten()
  homogeneity = graycoprops(glcm, "homogeneity").flatten()
  energy = graycoprops(glcm, "energy").flatten()
  correlation = graycoprops(glcm, "correlation").flatten()

  return np.hstack([contrast, homogeneity, energy, correlation])

In [8]:
def extract_all_features(loader):
    glcm_feat = []
    mobile_feat = []
    google_feat = []
    conv_feat = []
    labels = []

    with torch.no_grad():
        for images, lbl in loader:
            images  = images.to(device)

            for img in images:
                glcm_feat.append(get_glcm_features(img))

            x1 = mobilenet.features(images)
            x1 = mobilenet.avgpool(x1)
            x1 = torch.flatten(x1, 1)

            x2 = googlenet.conv1(images)
            x2 = googlenet.maxpool1(x2)
            x2 = googlenet.conv2(x2)
            x2 = googlenet.conv3(x2)
            x2 = googlenet.maxpool2(x2)
            x2 = googlenet.inception3a(x2)
            x2 = googlenet.inception3b(x2)
            x2 = googlenet.maxpool3(x2)
            x2 = googlenet.inception4a(x2)
            x2 = googlenet.inception4b(x2)
            x2 = googlenet.inception4c(x2)
            x2 = googlenet.inception4d(x2)
            x2 = googlenet.inception4e(x2)
            x2 = googlenet.maxpool4(x2)
            x2 = googlenet.inception5a(x2)
            x2 = googlenet.inception5b(x2)
            x2 = googlenet.avgpool(x2)
            x2 = torch.flatten(x2, 1)

            x3 = convnext.features(images)
            x3 = convnext.avgpool(x3)
            x3 = torch.flatten(x3, 1)

            mobile_feat.append(x1.cpu().numpy())
            google_feat.append(x2.cpu().numpy())
            conv_feat.append(x3.cpu().numpy())
            labels.append(lbl.numpy())

    glcm_feat = np.array(glcm_feat)
    mobile_feat = np.vstack(mobile_feat)
    google_feat = np.vstack(google_feat)
    conv_feat = np.vstack(conv_feat)
    labels = np.hstack(labels)

    return glcm_feat, mobile_feat, google_feat, conv_feat, labels

# Extracting Features

In [9]:
NOISED_PROC  = "../dataset/noised_test_processed"

for variant in VARIANTS:
    print("=" * 55)
    print(f"  Variant : {variant}")
    print("=" * 55)

    test_dir = os.path.join(NOISED_PROC, variant)
    save_dir = os.path.join("../features/ensemble_noise", variant)

    test_dataset = datasets.ImageFolder(test_dir, transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    print(f"  Images  : {len(test_dataset)}  |  Batches : {len(test_loader)}")

    glcm, mobile, google, conv, labels = extract_all_features(test_loader)

    X_noise_test = np.hstack([glcm, mobile, google, conv])
    y_noise_test = labels

    print(f"  GLCM        : {glcm.shape}")
    print(f"  MobileNetV3 : {mobile.shape}")
    print(f"  GoogLeNet   : {google.shape}")
    print(f"  ConvNeXt    : {conv.shape}")
    print(f"  X_noise_test: {X_noise_test.shape}")
    print(f"  y_noise_test: {y_noise_test.shape}")

    np.save(os.path.join(save_dir, "X_noise_test.npy"), X_noise_test)
    np.save(os.path.join(save_dir, "y_noise_test.npy"), y_noise_test)
    print(f" Saved in ensemble_noise/{variant}/\n")

print("All variants extracted and saved")

  Variant : salt_pepper_0.03
  Images  : 4380  |  Batches : 137
  GLCM        : (4380, 32)
  MobileNetV3 : (4380, 576)
  GoogLeNet   : (4380, 1024)
  ConvNeXt    : (4380, 768)
  X_noise_test: (4380, 2400)
  y_noise_test: (4380,)
 Saved in ensemble_noise/salt_pepper_0.03/

  Variant : salt_pepper_0.05
  Images  : 4380  |  Batches : 137
  GLCM        : (4380, 32)
  MobileNetV3 : (4380, 576)
  GoogLeNet   : (4380, 1024)
  ConvNeXt    : (4380, 768)
  X_noise_test: (4380, 2400)
  y_noise_test: (4380,)
 Saved in ensemble_noise/salt_pepper_0.05/

  Variant : salt_pepper_0.07
  Images  : 4380  |  Batches : 137
  GLCM        : (4380, 32)
  MobileNetV3 : (4380, 576)
  GoogLeNet   : (4380, 1024)
  ConvNeXt    : (4380, 768)
  X_noise_test: (4380, 2400)
  y_noise_test: (4380,)
 Saved in ensemble_noise/salt_pepper_0.07/

  Variant : gaussian
  Images  : 4380  |  Batches : 137
  GLCM        : (4380, 32)
  MobileNetV3 : (4380, 576)
  GoogLeNet   : (4380, 1024)
  ConvNeXt    : (4380, 768)
  X_noise_tes

In [10]:
model_dense = models.densenet121()
in_features = model_dense.classifier.in_features
model_dense.classifier = nn.Linear(in_features, NUM_CLASSES)

model_dense.load_state_dict(torch.load(os.path.join(MODEL_PATH, "densenet121.pth"), map_location=device))

model_dense = model_dense.to(device)
model_dense.eval()
print("DenseNet121 loaded")

DenseNet121 loaded


In [11]:
def extract_features(loader,model):
    model.eval()

    features = []
    preds    = []

    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device)

            x = model.features(images)
            x = nn.functional.relu(x)
            x = nn.functional.adaptive_avg_pool2d(x, (1, 1))
            x = torch.flatten(x, 1)

            out = model.classifier(x)
            prob = torch.softmax(out, 1)

            features.append(x.cpu().numpy())
            preds.append(prob.cpu().numpy())

    features = np.vstack(features)
    preds = np.vstack(preds)
    return features, preds

In [12]:
MODEL_PATH = "../models"
DENSE_PATH = "../features/dense_noise"     

for v in VARIANTS:
    os.makedirs(os.path.join(DENSE_PATH, v), exist_ok=True)
    print(f"Ready : {DENSE_PATH}/{v}")


Ready : ../features/dense_noise/salt_pepper_0.03
Ready : ../features/dense_noise/salt_pepper_0.05
Ready : ../features/dense_noise/salt_pepper_0.07
Ready : ../features/dense_noise/gaussian
Ready : ../features/dense_noise/masking


In [13]:
for variant in VARIANTS:
    print("=" * 55)
    print(f"  Variant : {variant}")
    print("=" * 55)

    test_dir = os.path.join(NOISED_PROC, variant)
    save_dir = os.path.join("../features/dense_noise",   variant)

    test_dataset = datasets.ImageFolder(test_dir,  transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    print(f"  Images  : {len(test_dataset)}  |  Batches : {len(test_loader)}")

    features, preds = extract_features(test_loader,model_dense)

    print(f"  Features : {features.shape}")
    print(f"  Preds    : {preds.shape}")

    np.save(os.path.join(save_dir, "X_dense_noise_test.npy"), features)
    np.save(os.path.join(save_dir, "y_dense_noise_test.npy"), preds)
    print(f"  Saved dense_noise/{variant}/\n")

print("All variants extracted and saved")

  Variant : salt_pepper_0.03
  Images  : 4380  |  Batches : 137
  Features : (4380, 1024)
  Preds    : (4380, 38)
  Saved dense_noise/salt_pepper_0.03/

  Variant : salt_pepper_0.05
  Images  : 4380  |  Batches : 137
  Features : (4380, 1024)
  Preds    : (4380, 38)
  Saved dense_noise/salt_pepper_0.05/

  Variant : salt_pepper_0.07
  Images  : 4380  |  Batches : 137
  Features : (4380, 1024)
  Preds    : (4380, 38)
  Saved dense_noise/salt_pepper_0.07/

  Variant : gaussian
  Images  : 4380  |  Batches : 137
  Features : (4380, 1024)
  Preds    : (4380, 38)
  Saved dense_noise/gaussian/

  Variant : masking
  Images  : 4380  |  Batches : 137
  Features : (4380, 1024)
  Preds    : (4380, 38)
  Saved dense_noise/masking/

All variants extracted and saved
